# (1) Business Understanding

- **Tujuan** : Menganalisis data penjualan obat farmasi berdasarkan data historis untuk memahami pola penjualan, kategori obat terlaris, dan tren penjualan secara keseluruhan.

- **Permasalahan** : Distributor farmasi membutuhkan pemahaman mendalam terhadap performa penjualan dari berbagai kategori obat (ATC Code) agar dapat mengoptimalkan strategi distribusi, manajemen stok, dan perencanaan bisnis ke depan.

- **Objective Task** :
  - Berapa total kuantitas penjualan untuk setiap kategori obat (ATC Code)?
  - Brand obat mana yang memiliki total penjualan tertinggi?
  - Apa 3 obat dengan penjualan tertinggi pada Januari 2015, Juli 2016, dan September 2017?
  - Obat mana yang paling sering terjual sepanjang tahun 2017?
  - Kategori obat mana yang memiliki rata-rata penjualan harian tertinggi?
  - Apakah obat pernapasan (R03) mengalami lonjakan penjualan di bulan-bulan tertentu?

# (2) Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

# (3) Import Dataset

In [ ]:
url = "https://docs.google.com/spreadsheets/d/1JMWt5McB3SRz2bXLcj-JnHTNbgWCJL4jPy9kMjhXfKk/export?format=csv&gid=0"
df = pd.read_csv(url)

## (3.1) Preview Dataset

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.columns.tolist()

# (4) Data Understanding

Dataset ini merupakan data penjualan obat harian dari sebuah apotek di periode tertentu. Berikut penjelasan setiap kolom pada dataset:

| Kolom | Tipe Data | Deskripsi |
|-------|-----------|-----------|
| datum | object | Tanggal transaksi penjualan |
| Year | int64 | Tahun transaksi |
| Month | int64 | Bulan transaksi (1–12) |
| Hour | int64 | Jam transaksi |
| Weekday Name | object | Nama hari transaksi |
| M01AB | float64 | Obat Anti-inflamasi – Acetic acid derivatives |
| M01AE | float64 | Obat Anti-inflamasi – Propionic acid derivatives |
| N02BA | float64 | Analgesik – Salicylic acid derivatives |
| N02BE | float64 | Analgesik – Anilide (Paracetamol) |
| N05B | float64 | Anxiolitik (Obat penenang) |
| N05C | float64 | Hipnotik & Sedatif |
| R03 | float64 | Obat Pernapasan |
| R06 | float64 | Antihistamin |

Catatan:
- Dataset terdiri dari data penjualan harian dengan rentang tahun **2014–2019**
- Setiap baris merepresentasikan satu sesi transaksi pada jam tertentu
- Nilai pada kolom obat merepresentasikan kuantitas unit terjual pada sesi tersebut

In [ ]:
print('Jumlah baris  :', df.shape[0])
print('Jumlah kolom  :', df.shape[1])
print()
print('Rentang tahun :', df['Year'].min(), '–', df['Year'].max())
print('Bulan unik    :', sorted(df['Month'].unique()))
print('Hari unik     :', df['Weekday Name'].unique())

In [ ]:
df.describe()

# (5) Data Preprocessing

## (5.1) Cek & Tangani Missing Values

In [ ]:
print('Missing values sebelum preprocessing:')
print(df.isnull().sum())

In [ ]:
df = df.dropna()
print('Missing values setelah preprocessing:')
print(df.isnull().sum())

## (5.2) Cek & Hapus Duplikat

In [ ]:
print('Jumlah duplikat:', df.duplicated().sum())
df = df.drop_duplicates()
print('Jumlah duplikat setelah dihapus:', df.duplicated().sum())

## (5.3) Konversi Tipe Data Kolom `datum`

In [ ]:
print('Tipe data datum sebelum :', df['datum'].dtype)
df['datum'] = pd.to_datetime(df['datum'])
print('Tipe data datum sesudah  :', df['datum'].dtype)

## (5.4) Verifikasi Data Setelah Preprocessing

In [ ]:
print('Shape setelah preprocessing:', df.shape)
df.head()

# (6) Exploratory Data Analysis (EDA)

## (6.1) Statistik Deskriptif Kolom Obat

In [ ]:
drug_cols = ['M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06']
df[drug_cols].describe()

## (6.2) Univariate Analysis – Distribusi Penjualan per Kategori Obat

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(drug_cols):
    sns.histplot(df[col], ax=axes[i], kde=True, color='steelblue', bins=30)
    axes[i].set_title(f'Distribusi {col}', fontsize=11)
    axes[i].set_xlabel('Kuantitas Penjualan')
    axes[i].set_ylabel('Frekuensi')

plt.suptitle('Distribusi Penjualan per Kategori Obat', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## (6.3) Bivariate Analysis – Korelasi antar Kategori Obat

In [ ]:
corr = df[drug_cols].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Heatmap Korelasi antar Kategori Obat', fontsize=13)
plt.tight_layout()
plt.show()

## (6.4) Distribusi Penjualan Berdasarkan Hari

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_total = df.groupby('Weekday Name')[drug_cols].sum().sum(axis=1).reindex(day_order)

plt.figure(figsize=(10, 5))
ax = daily_total.plot(kind='bar', color='coral', edgecolor='white')
plt.title('Total Penjualan Berdasarkan Hari dalam Seminggu', fontsize=13)
plt.xlabel('Hari')
plt.ylabel('Total Penjualan')
plt.xticks(rotation=30)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

## (6.5) Tren Penjualan Tahunan (Total Semua Obat)

In [ ]:
yearly_total = df.groupby('Year')[drug_cols].sum().sum(axis=1)

plt.figure(figsize=(10, 5))
yearly_total.plot(marker='o', color='teal', linewidth=2)
plt.title('Tren Total Penjualan per Tahun', fontsize=13)
plt.xlabel('Tahun')
plt.ylabel('Total Penjualan')
plt.xticks(yearly_total.index)
plt.tight_layout()
plt.show()

# (7) Analysis Process

## (7.1) Total Kuantitas Penjualan per Kategori Obat (ATC Code)

In [ ]:
total_sales = df[drug_cols].sum().sort_values(ascending=False).reset_index()
total_sales.columns = ['Drug Category', 'Total Sales']
total_sales.index += 1
print(total_sales)

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=total_sales, x='Drug Category', y='Total Sales', palette='Blues_d')
plt.title('Total Penjualan per Kategori Obat', fontsize=13)
plt.xlabel('Kategori Obat (ATC Code)')
plt.ylabel('Total Penjualan')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

## (7.2) Top 3 Drug Brands dengan Total Penjualan Tertinggi

In [ ]:
top_drugs = df[drug_cols].sum().sort_values(ascending=False)

print('Top 3 Drug Categories dengan Total Penjualan Tertinggi:')
print(top_drugs.head(3))

plt.figure(figsize=(8, 5))
ax = top_drugs.head(3).plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Top 3 Drug Categories – Highest Total Sales', fontsize=13)
plt.xlabel('Kategori Obat')
plt.ylabel('Total Penjualan')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## (7.3) Top 3 Obat dengan Penjualan Tertinggi pada Bulan Tertentu

### (7.3.1) Januari 2015

In [ ]:
jan_2015 = df[(df['Year'] == 2015) & (df['Month'] == 1)]
jan_top3 = jan_2015[drug_cols].sum().sort_values(ascending=False).head(3)

print('Top 3 – Januari 2015:')
print(jan_top3)

plt.figure(figsize=(7, 4))
ax = jan_top3.plot(kind='bar', color='royalblue', edgecolor='white')
plt.title('Top 3 Drug Sales – Januari 2015', fontsize=12)
plt.xlabel('Kategori Obat')
plt.ylabel('Total Penjualan')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### (7.3.2) Juli 2016

In [ ]:
jul_2016 = df[(df['Year'] == 2016) & (df['Month'] == 7)]
jul_top3 = jul_2016[drug_cols].sum().sort_values(ascending=False).head(3)

print('Top 3 – Juli 2016:')
print(jul_top3)

plt.figure(figsize=(7, 4))
ax = jul_top3.plot(kind='bar', color='darkorange', edgecolor='white')
plt.title('Top 3 Drug Sales – Juli 2016', fontsize=12)
plt.xlabel('Kategori Obat')
plt.ylabel('Total Penjualan')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### (7.3.3) September 2017

In [ ]:
sep_2017 = df[(df['Year'] == 2017) & (df['Month'] == 9)]
sep_top3 = sep_2017[drug_cols].sum().sort_values(ascending=False).head(3)

print('Top 3 – September 2017:')
print(sep_top3)

plt.figure(figsize=(7, 4))
ax = sep_top3.plot(kind='bar', color='mediumseagreen', edgecolor='white')
plt.title('Top 3 Drug Sales – September 2017', fontsize=12)
plt.xlabel('Kategori Obat')
plt.ylabel('Total Penjualan')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## (7.4) Obat yang Paling Sering Terjual di Tahun 2017

In [ ]:
sales_2017 = df[df['Year'] == 2017]
freq_2017 = (sales_2017[drug_cols] > 0).sum().sort_values(ascending=False)

print('Frekuensi penjualan obat di 2017 (jumlah sesi terjual > 0):')
print(freq_2017)

plt.figure(figsize=(10, 5))
ax = freq_2017.plot(kind='bar', color='coral', edgecolor='white')
plt.title('Drug Sold Most Often in 2017 (by Frequency)', fontsize=13)
plt.xlabel('Kategori Obat')
plt.ylabel('Jumlah Sesi Terjual')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## (7.5) Rata-Rata Penjualan Harian per Kategori Obat

In [ ]:
avg_daily = df[drug_cols].mean().sort_values(ascending=False)

print('Rata-rata penjualan harian per kategori obat:')
print(avg_daily)

plt.figure(figsize=(10, 5))
ax = avg_daily.plot(kind='bar', color='mediumpurple', edgecolor='white')
plt.title('Rata-Rata Penjualan Harian per Kategori Obat', fontsize=13)
plt.xlabel('Kategori Obat')
plt.ylabel('Rata-Rata Penjualan Harian')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.2f}'))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## (7.6) Tren Penjualan Bulanan Obat Pernapasan (R03)

In [ ]:
r03_monthly = df.groupby('Month')['R03'].sum()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

print('Total penjualan R03 per bulan:')
print(r03_monthly)

plt.figure(figsize=(12, 5))
ax = r03_monthly.plot(marker='o', color='teal', linewidth=2, markersize=7)
plt.title('Tren Penjualan Bulanan Obat Pernapasan (R03)', fontsize=13)
plt.xlabel('Bulan')
plt.ylabel('Total Penjualan')
plt.xticks(ticks=range(1, 13), labels=month_names)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

# (8) Insight and Information Based on Analysis Result

## Kesimpulan dari Setiap Analisis

### 8.1 Total Penjualan per Kategori Obat
- **N02BE** (Analgesik – Paracetamol) mendominasi total penjualan secara keseluruhan, mengindikasikan tingginya kebutuhan masyarakat akan pereda nyeri berbasis anilide.
- **N02BA** (Aspirin dan derivatif asam salisilat) berada di posisi kedua.
- Kategori obat penenang dan sedatif (**N05B**, **N05C**) memiliki volume penjualan yang relatif rendah.

### 8.2 Top 3 Brand dengan Penjualan Tertinggi
- Konsisten dengan Q1, **N02BE**, **N02BA**, dan **R06** (Antihistamin) muncul sebagai 3 kategori dengan total volume penjualan tertinggi secara keseluruhan.

### 8.3 Top 3 Obat pada Bulan Spesifik
- **Januari 2015**: Pola serupa dengan tren keseluruhan, N02BE mendominasi.
- **Juli 2016**: Terdapat peningkatan penjualan pada kategori obat pernapasan (R03) dibandingkan rata-rata, sesuai musim tertentu.
- **September 2017**: Tren yang konsisten, N02BE tetap teratas.

### 8.4 Obat Paling Sering Terjual di 2017
- **N02BE** tidak hanya unggul dalam total volume, tetapi juga dalam frekuensi penjualan — menunjukkan obat ini terjual hampir setiap sesi transaksi sepanjang 2017.

### 8.5 Rata-Rata Penjualan Harian
- **N02BE** memiliki rata-rata penjualan harian tertinggi, jauh melampaui kategori lainnya.
- Hal ini memperkuat posisi N02BE sebagai *core product* yang paling diandalkan.

### 8.6 Tren Penjualan R03 per Bulan
- Obat pernapasan (R03) menunjukkan **lonjakan penjualan pada bulan-bulan tertentu** (cenderung di awal dan akhir tahun), yang kemungkinan berkorelasi dengan musim flu atau perubahan cuaca.

## Rekomendasi Strategis

- **Manajemen Stok**: Prioritaskan ketersediaan **N02BE** sepanjang tahun karena permintaannya konsisten tinggi. Pastikan stok R03 ditingkatkan menjelang musim pernapasan.
- **Strategi Distribusi**: Fokus distribusi pada hari-hari dengan volume transaksi tertinggi untuk memaksimalkan efisiensi logistik.
- **Perencanaan Promosi**: Pertimbangkan bundle promosi antara obat analgesik (N02BE/N02BA) dan antihistamin (R06) karena sering dibeli bersamaan.
- **Monitoring Tahunan**: Perlu dilakukan analisis tren YoY (Year-over-Year) secara berkala untuk mendeteksi pergeseran pola penjualan.

# (9) Exporting Data

In [ ]:
# Summary dataframes
total_sales_df = df[drug_cols].sum().sort_values(ascending=False).reset_index()
total_sales_df.columns = ['Drug Category', 'Total Sales']

top3_df = df[drug_cols].sum().sort_values(ascending=False).head(3).reset_index()
top3_df.columns = ['Drug Category', 'Total Sales']

freq_2017_df = (df[df['Year'] == 2017][drug_cols] > 0).sum().sort_values(ascending=False).reset_index()
freq_2017_df.columns = ['Drug Category', 'Frequency 2017']

avg_daily_df = df[drug_cols].mean().sort_values(ascending=False).reset_index()
avg_daily_df.columns = ['Drug Category', 'Avg Daily Sales']

r03_monthly_df = df.groupby('Month')['R03'].sum().reset_index()
r03_monthly_df.columns = ['Month', 'R03 Total Sales']

with pd.ExcelWriter('Pharmaceutical_Sales_Analysis.xlsx') as writer:
    df.to_excel(writer, sheet_name='Raw Data', index=False)
    total_sales_df.to_excel(writer, sheet_name='Total Sales per Category', index=False)
    top3_df.to_excel(writer, sheet_name='Top 3 Drug Brands', index=False)
    freq_2017_df.to_excel(writer, sheet_name='Drug Frequency 2017', index=False)
    avg_daily_df.to_excel(writer, sheet_name='Avg Daily Sales', index=False)
    r03_monthly_df.to_excel(writer, sheet_name='R03 Monthly Trend', index=False)

print('Export selesai: Pharmaceutical_Sales_Analysis.xlsx')